In [ ]:
import cv2
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

app = FastAPI()

def frame_generator():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        ret, buffer = cv2.imencode(".jpg", frame)
        frame_bytes = buffer.tobytes()

        yield (b"--frame\r\n"
               b"Content-Type: image/jpeg\r\n\r\n" + frame_bytes + b"\r\n")

@app.get("/video")
def video_feed():
    return StreamingResponse(frame_generator(),
                             media_type="multipart/x-mixed-replace; boundary=frame")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server)
server_thread.daemon = True
server_thread.start()


In [ ]:
import cv2
from ultralytics import YOLO

# Set the URL for the video stream
url = "http://10.42.0.23/"  

# Load the YOLO model from the ONNX file
model = YOLO("/home/user/Documents/TverEyNg/backend/models/yolo11n.onnx")

# Open the video stream
cap = cv2.VideoCapture(url)

if not cap.isOpened():
    print("Cannot open stream")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    
    # Run object detection on the current frame
    results = model(frame, conf=0.3, iou=0.5)  # Use predict() for detection on a single frame

    # Draw results on the frame
    frame_with_results = results[0].plot()  # Plot detection results on the frame

    # Display the frame with detections
    cv2.imshow("YOLO Detection", frame_with_results)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and close windows
cap.release()
cv2.destroyAllWindows()


Forward Local Camera to global

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import requests

app = FastAPI()

ESP32_URL = "http://172.23.35.212:81/stream"


def stream_mjpeg():
    """Stream the MJPEG feed from ESP32 and forward it."""
    with requests.get(ESP32_URL, stream=True) as r:
        for chunk in r.iter_content(chunk_size=1024):
            if chunk:
                yield chunk


@app.get("/stream")
async def stream():
    return StreamingResponse(
        stream_mjpeg(),
        media_type="multipart/x-mixed-replace; boundary=frame"
    )
